# World Cup 2026 Match Predictions

This notebook builds leakage-safe match features, selects an expected-goals model using historical validation, optimizes company-game score picks, and runs the 48-team World Cup 2026 tournament simulator.

## 1. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/wdqgallego-git/worldcup-predictor.git"
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path("/content/worldcup-predictor")
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
for module_dir in (PROJECT_DIR / "src", PROJECT_DIR / "scripts"):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

print(f"Project directory: {PROJECT_DIR}")

## 2. Install requirements

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Requirements installed.")

## 3. Download data

In [ ]:
from download_data import download_all

manifest = download_all()
manifest

## 4. Load data

The official Round-of-32 third-place assignment matrix must be added before a strict final submission. Until then, tournament simulation prints a warning and uses the documented deterministic development fallback.

In [ ]:
from data_loader import load_fixtures, load_rankings, load_results
from run_final_predictions import load_third_place_assignment_matrix, validate_group_fixtures
from tournament_simulator import validate_2026_format

validate_2026_format()
results = load_results()
rankings = load_rankings()
fixtures = load_fixtures()
group_fixtures = validate_group_fixtures(fixtures)
third_place_matrix = load_third_place_assignment_matrix(require_official_matrix=False)

print(f"Historical matches: {len(results):,}")
print(f"Ranking rows: {len(rankings):,}")
print(f"2026 fixtures: {len(fixtures):,}")
print(f"Resolved group-stage fixtures: {len(group_fixtures):,}")
group_fixtures.head()

## 5. Build features

In [ ]:
from features import build_fixture_features, build_training_table

training_df, feature_cols = build_training_table(results, rankings)
fixture_features, _ = build_fixture_features(group_fixtures, results, rankings, feature_cols)

print(f"Training rows: {len(training_df):,}")
print(f"Feature count: {len(feature_cols)}")
fixture_features[["match_id", "team_a", "team_b", *feature_cols]].head()

## 6. Train expected-goals model

In [ ]:
from model import train_goal_models

models = train_goal_models(training_df, feature_cols)
print(f"Selected model: {models['selected_model_name']}")
models["validation_results"]

## 7. Predict fixtures

In [ ]:
from model import predict_expected_goals

fixture_predictions = predict_expected_goals(models, fixture_features)
fixture_predictions[["match_id", "team_a", "team_b", "expected_goals_a", "expected_goals_b"]].head(10)

## 8. Score probability matrix

In [ ]:
import pandas as pd

from config import MAX_GOALS_FINAL
from poisson import summarize_score_probs

example_match = fixture_predictions.iloc[0]
example_probabilities = summarize_score_probs(
    example_match["expected_goals_a"],
    example_match["expected_goals_b"],
    max_goals=MAX_GOALS_FINAL,
    method="independent",
)
print(f"{example_match['team_a']} vs {example_match['team_b']}")
print({key: value for key, value in example_probabilities.items() if key != "score_probs"})
pd.DataFrame(
    example_probabilities["score_probs"],
    index=[f"{example_match['team_a']} {goals}" for goals in range(MAX_GOALS_FINAL + 1)],
    columns=[f"{example_match['team_b']} {goals}" for goals in range(MAX_GOALS_FINAL + 1)],
)

## 9. Match expected-points optimizer

In [ ]:
from prediction_optimizer import summarize_match_strategy
from run_final_predictions import optimize_fixture_predictions, select_final_prediction_columns

example_strategy = summarize_match_strategy(
    example_probabilities["score_probs"],
    max_goals=MAX_GOALS_FINAL,
)
print("Safe pick:", example_strategy["safe_prediction"])
print("Aggressive pick:", example_strategy["aggressive_prediction"])
pd.DataFrame(example_strategy["top_candidates"])

final_predictions = select_final_prediction_columns(optimize_fixture_predictions(fixture_predictions))
final_predictions.head(10)

## 10. 2026 tournament simulation

Use `1_000` simulations while developing. Increase `N_SIMULATIONS` to `20_000` for the formal final run.

In [ ]:
from tournament_simulator import run_monte_carlo_tournament

N_SIMULATIONS = 1_000
output_dir = Path("outputs")
tournament_outputs = run_monte_carlo_tournament(
    fixtures=fixtures,
    match_predictions=fixture_predictions,
    n_simulations=N_SIMULATIONS,
    third_place_assignment_matrix=third_place_matrix,
    output_dir=output_dir,
)
tournament_outputs["champion_probabilities"].head(10)

## 11. Export final match and tournament outputs

In [ ]:
from run_final_predictions import save_match_predictions

save_match_predictions(final_predictions, output_dir)
export_files = [
    "final_predictions.csv",
    "final_predictions.xlsx",
    "team_path_simulations.csv",
    "champion_probabilities.csv",
    "runner_up_probabilities.csv",
    "team_tournament_probabilities.csv",
]
for file_name in export_files:
    path = output_dir / file_name
    print(f"{path}: {path.exists()}")

# Optional in Google Colab:
# from google.colab import files
# files.download(str(output_dir / "final_predictions.xlsx"))